### First set of imports 

In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
import os
import json
import torch 
import torch.nn as nn
import numpy as np

pd.set_option("display.max_colwidth", None)



### Convert videos json to df

In [2]:
#Get the videos json
videos = os.path.join("..", "Videos.json")

#Open the json
with open(videos, "r", encoding="utf-8") as f:
    data = json.load(f)

#Turn the json into a pandas dataframe with proper headings     
df = pd.json_normalize(data["videos"])

print(df.columns)
print(df.head())

### Get all the handlandmark paths 

Index(['id', 'Sign', 'filepath', 'HamNoSys', 'concept_url', 'HandLandmark'], dtype='str')
   id      Sign  \
0   1     April   
1   2    Athens   
2   3    August   
3   4    Berlin   
4   5  February   

                                                             filepath  \
0     C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   
1    C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\Athens.mp4   
2    C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\August.mp4   
3    C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\Berlin.mp4   
4  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\February.mp4   

                                          HamNoSys  \
0     
1                 
2         
3                                         
4                             

                               

### Get hand landmark paths

In [3]:
#find the path to each landmark json in the videos json
Hand_Landmarks_paths = df["HandLandmark"]

Hand_Landmarks = [p for p in Hand_Landmarks_paths if os.path.exists(p)]

print(Hand_Landmarks_paths)


0           C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json
1          C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\Athens_hands.json
2          C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\August_hands.json
3          C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\Berlin_hands.json
4        C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\February_hands.json
                                          ...                                    
1003          C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\yes_hands.json
1004    C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\yesterday_hands.json
1005          C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\you_hands.json
1006        C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\young_hands.json
1007         C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\zero_hands.json
Name: HandLandmark, Length: 1008, dtype: str


### Use all those paths to get the jsons and convert them to df

In [4]:

#Empty list for storing each dataframe of the handlandmarks
Hand_landmarks_dfs = []

for path in Hand_Landmarks:
    
    with open(path, "r") as f:
        data = json.load(f)
        
    #Create dataframes
    temp_df = pd.json_normalize(data)
    temp_df["source"] = path
    
    Hand_landmarks_dfs.append(temp_df)

### Combine all those dfs into one big df

In [5]:
#Take each handlandmark df and make it into one big df
Hand_Landmark_df = pd.concat(Hand_landmarks_dfs, ignore_index=True)

In [6]:
#Hand_Landmark_df.head(1)



### Create df with all the frames, landmarks, timings ect.

In [7]:
rows = []

#For each video
for _, video_row in Hand_Landmark_df.iterrows():
    video_path = video_row["video_path"]#Get each video path
    fps = video_row["fps"]#Get the fps for those videos
    source = video_row.get("source",None)#
    
    #for all the frames in current video
    for frame in video_row["frames"]:
        frame_index = frame["frame_index"] #Set frame index
        time_sec = frame["time_sec"]#Set the time of frame
        
        #Then for each hand in current frame
        for hand in frame["hands"]:
            handedness = hand["handedness"]#which hand
            handedness_score = hand["handedness_score"]#How sure that this is the right hand
            hand_index = hand["hand_index"]#The index of hand
            
            #For each landmark in hand
            for lm in hand["landmarks"]:
                rows.append({
                    "video_path": video_path,
                    "fps": fps,
                    "source": source,
                    "frame_index": frame_index,
                    "time_sec": time_sec,
                    "hand_index": hand_index,
                    "handedness": handedness,
                    "handedness_score": handedness_score,
                    "landmark_id": lm["id"],
                    "landmark_name": lm["name"],
                    "x": lm["x"],
                    "y": lm["y"],
                    "z": lm["z"],
                })
                

Hand_landmarks_df = pd.DataFrame(rows)

#print(rows[2])
#print(Hand_landmarks_df.head())

In [8]:
print(Hand_landmarks_df.columns)

Index(['video_path', 'fps', 'source', 'frame_index', 'time_sec', 'hand_index',
       'handedness', 'handedness_score', 'landmark_id', 'landmark_name', 'x',
       'y', 'z'],
      dtype='str')


### Convert hand landmark df to df seperated by frames and not landmark (will work better for models such as LSTM)

In [9]:
#Need to piivot table so that it is done by frame instead of landmark

wide_df = Hand_landmarks_df.pivot_table(
    index=["video_path", "frame_index", "time_sec", "source"],
    columns=["handedness", "landmark_name"],
    values=["x", "y", "z"]
)

#Create readable headings
wide_df.columns = [
    f"{hand}_{landmark}_{axis}"
    for axis, hand, landmark in wide_df.columns
]

wide_df = wide_df.reset_index()
wide_df = wide_df.fillna(0.0)

In [10]:
print(wide_df.head())
print(wide_df.shape)

                                                        video_path  \
0  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   
1  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   
2  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   
3  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   
4  C:\Users\mccor\Documents\Y4\Dis\Project\Data\External\April.mp4   

   frame_index  time_sec  \
0            0      0.00   
1            1      0.04   
2            5      0.20   
3            6      0.24   
4            7      0.28   

                                                                  source  \
0  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
1  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
2  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
3  C:\Users\mccor\Documents\Y4\Dis\Project\Data\Interim\April_hands.json   
4  C:\Users\mccor\Documents\Y4

###  breaks down that df into features

In [11]:
feature_cols = [col for col in wide_df.columns
                if col not in ["video_path", "frame_index", "time_sec","source"]]

sequences = []
video_ids = []

for video, group in wide_df.groupby("video_path"):
    
    group = group.sort_values("frame_index")
    seq = group[feature_cols].values
    sequences.append(seq)
    video_ids.append(video)
    
print(len(sequences)) 
print(sequences[0].shape)

    

1008
(239, 126)


###  pad out video frames to match the longets video (may need to do some adjusting for models)

In [12]:
max_len = max(seq.shape[0] for seq in sequences)
num_features = sequences[0].shape[1]

x = np.zeros((len(sequences), max_len, num_features))

for i, seq in enumerate(sequences):
    length = seq.shape[0]
    x[i, :length, :] = seq

print(x.shape)
    
    

(1008, 1951, 126)


### Convert into a tensor object

In [13]:
X_tensor = torch.tensor(x, dtype=torch.float32)

print(X_tensor.shape)

torch.Size([1008, 1951, 126])


In [14]:

video_dict = {
    os.path.basename(fp): ham
    for fp, ham in zip(df["filepath"], df["HamNoSys"])
}
print(video_dict)

{'April.mp4': '\ue0e8\ue0e2\ue002\ue00d\ue0e7\ue002\ue00c\ue0e3\ue0e2\ue027\ue03d\ue0e7\ue031\ue03c\ue0e6\ue03b\ue0e3\ue0e2\ue070\ue077\ue0e7\ue071\ue075\ue0e3\ue0d1\ue051\ue0e2\ue0aa\ue006\ue03c\ue0e7\ue0aa\ue002\ue00d\ue03a\ue0e3\ue080\ue0c6\ue0bb\ue0e2\ue071\ue077\ue0e7\ue071\ue075\ue0e3\ue0d1', 'Athens.mp4': '\ue0e9\ue0e2\ue000\ue00c\ue028\ue03b\ue0e7\ue001\ue00c\ue02a\ue038\ue0e3\ue0e2\ue067\ue0e7\ue075\ue070\ue0e3\ue0d0\ue084\ue0c6\ue0e2\ue067\ue0e7\ue075\ue070\ue0e3\ue0d1\ue0e2\ue080\ue0c6\ue0e7\ue0af\ue0e3\ue0d8', 'August.mp4': '\ue0e8\ue0dd\ue0e2\ue002\ue00d\ue0e7\ue001\ue00c\ue0e3\ue0e2\ue031\ue028\ue03d\ue0e7\ue031\ue03a\ue0e3\ue0e2\ue070\ue077\ue0e7\ue071\ue077\ue0e3\ue0d1\ue051\ue0aa\ue005\ue00d\ue0e0\ue097\ue0aa\ue000\ue00d\ue0e1\ue0e2\ue074\ue079\ue0e7\ue071\ue079\ue0e3\ue0d1', 'Berlin.mp4': '\ue001\ue00d\ue027\ue03e\ue042\ue0d1\ue0e2\ue0aa\ue021\ue03e\ue0e3', 'February.mp4': '\ue0e9\ue003\ue00d\ue028\ue03c\ue0e2\ue077\ue072\ue0e7\ue076\ue072\ue0e3\ue0d1\ue0e2\ue093\ue0a

In [15]:
y_raw = []

for vid in video_ids:
    filename = os.path.basename(vid)
    hamnosys = video_dict[filename]
    y_raw.append(hamnosys)
    
print(y_raw)

['\ue0e8\ue0e2\ue002\ue00d\ue0e7\ue002\ue00c\ue0e3\ue0e2\ue027\ue03d\ue0e7\ue031\ue03c\ue0e6\ue03b\ue0e3\ue0e2\ue070\ue077\ue0e7\ue071\ue075\ue0e3\ue0d1\ue051\ue0e2\ue0aa\ue006\ue03c\ue0e7\ue0aa\ue002\ue00d\ue03a\ue0e3\ue080\ue0c6\ue0bb\ue0e2\ue071\ue077\ue0e7\ue071\ue075\ue0e3\ue0d1', '\ue0e9\ue0e2\ue000\ue00c\ue028\ue03b\ue0e7\ue001\ue00c\ue02a\ue038\ue0e3\ue0e2\ue067\ue0e7\ue075\ue070\ue0e3\ue0d0\ue084\ue0c6\ue0e2\ue067\ue0e7\ue075\ue070\ue0e3\ue0d1\ue0e2\ue080\ue0c6\ue0e7\ue0af\ue0e3\ue0d8', '\ue0e8\ue0dd\ue0e2\ue002\ue00d\ue0e7\ue001\ue00c\ue0e3\ue0e2\ue031\ue028\ue03d\ue0e7\ue031\ue03a\ue0e3\ue0e2\ue070\ue077\ue0e7\ue071\ue077\ue0e3\ue0d1\ue051\ue0aa\ue005\ue00d\ue0e0\ue097\ue0aa\ue000\ue00d\ue0e1\ue0e2\ue074\ue079\ue0e7\ue071\ue079\ue0e3\ue0d1', '\ue001\ue00d\ue027\ue03e\ue042\ue0d1\ue0e2\ue0aa\ue021\ue03e\ue0e3', '\ue0e9\ue003\ue00d\ue028\ue03c\ue0e2\ue077\ue072\ue0e7\ue076\ue072\ue0e3\ue0d1\ue0e2\ue093\ue0aa\ue008\ue020\ue03d\ue0e3\ue076\ue070\ue0d1', '\ue009\ue00d\ue010\ue020

In [16]:
print(y_raw[0])
print(len(y_raw[0]))



47


In [17]:
all_tokens = set()

for seq in y_raw:
    for char in seq:
        all_tokens.add(char)

print(len(all_tokens))

176


In [18]:
token_to_id = {token: i+3 for i, token in enumerate(sorted(all_tokens))}

# special tokens
token_to_id["<PAD>"] = 0
token_to_id["<SOS>"] = 1
token_to_id["<EOS>"] = 2

vocab_size = len(token_to_id)

id_to_token = {i: t for t, i in token_to_id.items()}

print(list(token_to_id.items())[:10])

[('\ue000', 3), ('\ue001', 4), ('\ue002', 5), ('\ue003', 6), ('\ue004', 7), ('\ue005', 8), ('\ue006', 9), ('\ue007', 10), ('\ue008', 11), ('\ue009', 12)]


In [19]:
y_sequences = []

for seq in y_raw:
    encoded = [token_to_id["<SOS>"]]
    
    for char in seq:
        encoded.append(token_to_id[char])
        
    encoded.append(token_to_id["<EOS>"])
    
    y_sequences.append(encoded)

In [20]:
print(y_sequences[0])

[1, 176, 170, 5, 16, 175, 5, 15, 171, 170, 30, 46, 175, 40, 45, 174, 44, 171, 170, 83, 90, 175, 84, 88, 171, 157, 66, 170, 131, 9, 45, 175, 131, 5, 16, 43, 171, 94, 149, 141, 170, 84, 90, 175, 84, 88, 171, 157, 2]


In [21]:
max_len = max(len(seq) for seq in y_sequences)
print(max_len)

135


In [22]:

y_padded = np.zeros((len(y_sequences), max_len), dtype=int)

for i, seq in enumerate(y_sequences):
    y_padded[i, :len(seq)] = seq

In [23]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import Adam #Do I keep adam or GD

import lightning as L
from torch.utils.data import TensorDataset, DataLoader


In [24]:
y_tensor = torch.tensor(y_padded, dtype=torch.long)

print(X_tensor.shape)
print(y_tensor.shape)
print(len(token_to_id))

torch.Size([1008, 1951, 126])
torch.Size([1008, 135])
179


In [25]:
#Need to ammend sizes when adding other feautures in

class Encoder(nn.Module):
    def __init__(self,input_size = 126, hidden_size = 128):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        
    def forward(self, x):
        print(type(x), x.shape)
        outputs, (hidden,cell) = self.lstm(x)
        return hidden, cell

In [26]:
#Vocab size for number of hamnosys symbols( may need to adapt)
class Decoder(nn.Module):
    def __init__(self, vocab_size, hidden_size = 128,):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first = True)
        self.fc = nn.Linear(hidden_size, vocab_size)
        
    def forward(self, x, hidden, cell):
        x = self.embedding(x)
        outputs, (hidden,cell) = self.lstm(x, (hidden,cell))
        predictions = self.fc(outputs)
        return predictions, hidden, cell
        

In [27]:
class Seq2seq(nn.Module):
    def __init__(self, encoder,  decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        
    def forward(self, src, trg):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        vocab_size = self.decoder.fc.out_features
        
        outputs = torch.zeros(batch_size, trg_len ,vocab_size)
        
        hidden, cell = self.encoder(src)
        input_token = trg[:,0].unsqueeze(1)
        
        for t in range(1,trg_len):
            output, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs[:,t:t+1,:] = output
            
            input_token = trg[:,t].unsqueeze(1)
        return outputs
        

In [28]:
criterion = nn.CrossEntropyLoss(ignore_index=0)

In [29]:
encoder = Encoder(input_size=126, hidden_size=256)
decoder = Decoder(vocab_size=vocab_size, hidden_size=256)

model = Seq2seq(encoder, decoder)

In [30]:
optimiser = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):
    
    model.train()
    epoch_loss = 0
    
    for i in range(len(X_tensor)):
        
        src = X_tensor[i].unsqueeze(0)   # [1, frames, features]
        trg = y_tensor[i].unsqueeze(0)   # [1, seq_len]
        
        optimiser.zero_grad()
        
        output = model(src, trg)
        
        output = output[:, 1:, :].reshape(-1, output.shape[-1])
        trg = trg[:, 1:].reshape(-1)
        
        loss = criterion(output, trg)
        
        loss.backward()
        optimiser.step()
        
        epoch_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {epoch_loss:.4f}")

<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
<class 'torch.Tensor'> torch.Size([1, 1951, 126])
